# KLTN - SynapseML LightGBM

Notebook này được viết lại để khớp sát notebook LightGBM Python đã chạy thành công:

- Giữ cùng `RANDOM_STATE`, target batch, history offsets, sample size, split ratio, Optuna search space, anomaly levels, batch monitoring range, p-value settings.
- Chuyển engine từ pandas + Python LightGBM sang Spark DataFrame + SynapseML LightGBM.
- Không dùng `.toPandas()` cho training chính; chỉ collect các bảng summary nhỏ để hiển thị/báo cáo.
- SHAP dùng `featuresShapCol` của SynapseML và map lại `feature_index -> feature_name`.


# Import libraries


In [ ]:
import os
import json
import time
import gc
import warnings
from pyspark import StorageLevel
import time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, classification_report

import optuna

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from synapse.ml.lightgbm import LightGBMClassifier

print("Optuna version:", optuna.__version__)

# Config


In [ ]:
RANDOM_STATE = 42

# Sửa đường dẫn này theo notebook server / cụm của bạn
DATA_PATH = "/opt/spark/workspace/input/data/yellow_tripdata_2025-01.parquet"
OUTPUT_DIR = "/opt/spark/workspace/output/synapseml_experiment"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cột thời gian gốc trong TLC Yellow Taxi
PICKUP_COL = "tpep_pickup_datetime"
DROPOFF_COL = "tpep_dropoff_datetime"

# Batch mục tiêu cần kiểm tra - GIỮ GIỐNG NOTEBOOK LIGHTGBM
TARGET_PROCESSING_DATE_HOUR = pd.Timestamp("2025-01-20 00:00:00")

# Vì hệ thống ingest 12h/lần, batch liền trước là target - 12h
PREVIOUS_BATCH_OFFSET = pd.Timedelta(hours=12)

# Các mốc lịch sử bổ sung để so sánh
# Tất cả sẽ được gộp chung với batch liền trước thành label = 0
HISTORY_OFFSETS = [
    pd.Timedelta(days=1),
    pd.Timedelta(days=7),
    pd.Timedelta(days=14),
]

# Sampling - GIỮ GIỐNG NOTEBOOK LIGHTGBM
MIN_ROWS_PER_GROUP = 100
TARGET_SAMPLE_PER_GROUP = 10_000

# Nếu True: mỗi processing_date_hour trong nhóm label = 0 lấy tối đa TARGET_SAMPLE_PER_GROUP dòng
SAMPLE_EACH_COMPARISON_BATCH = True

# Train / valid / test split - GIỮ GIỐNG NOTEBOOK LIGHTGBM
TEST_SIZE = 0.15
VALID_SIZE = 0.15
VALID_SIZE_FROM_TRAIN = VALID_SIZE / (1 - TEST_SIZE)

# Optuna tuning - GIỮ GIỐNG NOTEBOOK LIGHTGBM
N_TRIALS = 100

# LightGBM / SynapseML LightGBM - GIỮ GIỐNG NOTEBOOK LIGHTGBM
# numIterations được cố định bằng N_ESTIMATORS_MAX; earlyStoppingRound sẽ dừng sớm.
N_ESTIMATORS_MAX = 2000
EARLY_STOPPING_ROUNDS = 100

# Run switches
RUN_OPTUNA = True

# Các phần này rất tốn thời gian trên SynapseML vì phải train/transform nhiều Spark jobs.
# Bật lại khi cần chạy đầy đủ thí nghiệm.
RUN_SYNTHETIC_EXPERIMENTS = False
RUN_BATCH_MONITORING = False

# SHAP trên SynapseML rất nặng vì phải transform có featuresShapCol.
# Giữ False để tối ưu tốc độ; bật True khi thật sự cần giải thích mô hình.
RUN_FINAL_SHAP = False
RUN_BATCH_SHAP = False

# ============================================================
# SynapseML LightGBM cluster profile
# Profile nhanh hơn theo benchmark của bạn cho cụm 2 workers, mỗi worker 6 cores / 12GB RAM:
# spark.executor.instances = 2
# spark.executor.cores     = 6
# spark.task.cpus          = 2
# => available task slots = 2 * floor(6/2) = 6
# => numTasks=6, numThreads=2, repartition(6)
#
# Lưu ý: Nếu SparkSession đã tồn tại, restart kernel/cluster trước khi chạy lại
# để spark.task.cpus được áp dụng đúng.
# ============================================================
LGBM_NUM_TASKS = 6
LGBM_NUM_THREADS = 2
LGBM_NUM_PARTITIONS = 6

# Spark preprocessing/shuffle profile.
# available_task_slots = 2 * floor(6 / 2) = 6
# Chọn 3 waves cho preprocessing: 6 slots * 3 = 18 partitions.
# LightGBM fit vẫn dùng repartition(6) để khớp numTasks=6.
SPARK_SQL_SHUFFLE_PARTITIONS = 18
SPARK_DEFAULT_PARALLELISM = 18

# Không cố định port khi chạy nhiều trial trong notebook, tránh lỗi port busy.
USE_FIXED_PORTS = False

print("DATA_PATH:", DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("TARGET_PROCESSING_DATE_HOUR:", TARGET_PROCESSING_DATE_HOUR)
print("TEST_SIZE:", TEST_SIZE, "VALID_SIZE:", VALID_SIZE)
print("N_TRIALS:", N_TRIALS)
print("N_ESTIMATORS_MAX:", N_ESTIMATORS_MAX)
print("EARLY_STOPPING_ROUNDS:", EARLY_STOPPING_ROUNDS)
print("SynapseML profile:", {
    "numTasks": LGBM_NUM_TASKS,
    "numThreads": LGBM_NUM_THREADS,
    "partitions": LGBM_NUM_PARTITIONS,
    "use_fixed_ports": USE_FIXED_PORTS,
})

# Spark session


In [ ]:
# Spark session
# Lưu ý: nếu SparkSession đã tồn tại, một số config cấp executor có thể không đổi được
# cho tới khi restart kernel / cluster.
spark = (
    SparkSession.builder
    .appName("SynapseML LightGBM")
    .master("spark://spark-master:7077")
    .config("spark.executor.instances", "2")
    .config("spark.executor.cores", "6")
    .config("spark.executor.memory", "6g")
    .config("spark.executor.memoryOverhead", "4g")
    .config("spark.task.cpus", str(LGBM_NUM_THREADS))  # profile 6x2 => task.cpus = 2
    .config("spark.sql.shuffle.partitions", str(SPARK_SQL_SHUFFLE_PARTITIONS))
    .config("spark.default.parallelism", str(SPARK_DEFAULT_PARALLELISM))
    .config("spark.dynamicAllocation.enabled", "false")
    .config("spark.speculation", "false")
    .config("spark.scheduler.mode", "FIFO")
    .config("spark.python.worker.reuse", "true")
    .config("spark.executorEnv.OMP_NUM_THREADS", str(LGBM_NUM_THREADS))
    .config("spark.executorEnv.MKL_NUM_THREADS", "1")
    .config("spark.executorEnv.OPENBLAS_NUM_THREADS", "1")
    .config("spark.executorEnv.NUMEXPR_NUM_THREADS", "1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

conf = spark.sparkContext.getConf()

def get_conf_int(key, default):
    try:
        return int(conf.get(key, str(default)))
    except Exception:
        return int(default)

executor_instances = get_conf_int("spark.executor.instances", 2)
executor_cores = get_conf_int("spark.executor.cores", 6)
task_cpus = get_conf_int("spark.task.cpus", LGBM_NUM_THREADS)
available_task_slots = executor_instances * (executor_cores // task_cpus)

print("Spark application:", spark.sparkContext.appName)
print("Spark master:", spark.sparkContext.master)
print("executor.instances:", executor_instances)
print("executor.cores:", executor_cores)
print("spark.task.cpus:", task_cpus)
print("Available Spark task slots:", available_task_slots)
print("Expected SynapseML numTasks:", LGBM_NUM_TASKS)
print("spark.sql.shuffle.partitions:", conf.get("spark.sql.shuffle.partitions"))
print("spark.default.parallelism:", conf.get("spark.default.parallelism"))

if available_task_slots < LGBM_NUM_TASKS:
    raise ValueError(
        f"Spark chỉ có {available_task_slots} task slots nhưng LGBM_NUM_TASKS={LGBM_NUM_TASKS}. "
        "Hãy giảm LGBM_NUM_TASKS hoặc giảm spark.task.cpus / tăng executor cores."
    )

# Data Loading


In [ ]:
df_raw = spark.read.parquet(DATA_PATH)

print("Raw row count:", df_raw.count())
df_raw.printSchema()
df_raw.show(5, truncate=False)

# Data Pre-processing


In [ ]:
df_raw = (
    df_raw
    .withColumn(PICKUP_COL, F.to_timestamp(F.col(PICKUP_COL)))
    .withColumn(DROPOFF_COL, F.to_timestamp(F.col(DROPOFF_COL)))
    .dropna(subset=[PICKUP_COL, DROPOFF_COL])
)

stats = df_raw.select(
    F.count("*").alias("row_count"),
    F.min(PICKUP_COL).alias("min_pickup"),
    F.max(PICKUP_COL).alias("max_pickup"),
    F.min(DROPOFF_COL).alias("min_dropoff"),
    F.max(DROPOFF_COL).alias("max_dropoff"),
)

stats.show(truncate=False)

In [ ]:
df_raw = df_raw.withColumn(
    "date_hour",
    F.date_trunc("hour", F.col(DROPOFF_COL))
)

# Vì file là dữ liệu tháng 01/2025, ta giữ các bản ghi có date_hour nằm trong tháng 01/2025
# GIỮ GIỐNG NOTEBOOK LIGHTGBM
JAN_START = "2025-01-01 00:00:00"
JAN_END = "2025-02-01 00:00:00"

df_raw = df_raw.filter(
    (F.col("date_hour") >= F.to_timestamp(F.lit(JAN_START))) &
    (F.col("date_hour") < F.to_timestamp(F.lit(JAN_END)))
)

stats = df_raw.select(
    F.count("*").alias("row_count"),
    F.min("date_hour").alias("min_date_hour"),
    F.max("date_hour").alias("max_date_hour"),
)

stats.show(truncate=False)


In [ ]:
# Giả lập batch ingest 12h/lần:
# - date_hour 00:00 -> 11:00 được xử lý tại cùng ngày 12:00
# - date_hour 12:00 -> 23:00 được xử lý tại ngày hôm sau 00:00
# Logic này tương đương compute_processing_date_hour trong notebook LightGBM.

df_raw = df_raw.withColumn(
    "processing_date_hour",
    F.when(
        F.hour("date_hour") < 12,
        F.date_trunc("day", F.col("date_hour")) + F.expr("INTERVAL 12 HOURS")
    ).otherwise(
        F.date_trunc("day", F.col("date_hour")) + F.expr("INTERVAL 1 DAY")
    )
)

stats = df_raw.select(
    F.min("processing_date_hour").alias("min_processing_date_hour"),
    F.max("processing_date_hour").alias("max_processing_date_hour"),
)

stats.show(truncate=False)

df_raw.select(
    "date_hour",
    "processing_date_hour"
).distinct().orderBy(
    "processing_date_hour",
    "date_hour"
).show(30, truncate=False)

In [ ]:
print("Schema after preprocessing:")
df_raw.printSchema()

df_raw.limit(5).show(truncate=False)

processing_counts = (
    df_raw
    .groupBy("processing_date_hour")
    .count()
    .withColumnRenamed("count", "row_count")
    .orderBy("processing_date_hour")
)

print("First 5 processing_date_hour:")
processing_counts.show(5, truncate=False)

print("Last 5 processing_date_hour:")
processing_counts.orderBy(
    F.desc("processing_date_hour")
).show(5, truncate=False)


# Target/history batch construction


In [ ]:
# Target batch: label = 1
target_processing_hour = pd.to_datetime(TARGET_PROCESSING_DATE_HOUR)

# Comparison batches: label = 0
# Bao gồm:
# - batch xử lý ngay trước đó
# - các batch lịch sử: hôm qua, 1 tuần trước, 2 tuần trước, ...
comparison_processing_hours = []
comparison_processing_hours.append(target_processing_hour - PREVIOUS_BATCH_OFFSET)
comparison_processing_hours.extend([target_processing_hour - offset for offset in HISTORY_OFFSETS])
comparison_processing_hours = sorted(set(comparison_processing_hours))

def ts_str(ts):
    return pd.Timestamp(ts).strftime("%Y-%m-%d %H:%M:%S")

target_ts = ts_str(target_processing_hour)
comparison_ts = [ts_str(h) for h in comparison_processing_hours]

target_count = df_raw.filter(F.col("processing_date_hour") == F.lit(target_ts)).count()

print("Target processing hour, label = 1:")
print("-", target_processing_hour, "| rows:", target_count)

print("\nComparison processing hours, label = 0:")
for h, h_str in zip(comparison_processing_hours, comparison_ts):
    cnt = df_raw.filter(F.col("processing_date_hour") == F.lit(h_str)).count()
    print("-", h, "| rows:", cnt)


In [ ]:
def sample_by_processing_hour_spark(
    source_df,
    processing_hour,
    label,
    max_rows,
    random_state
):
    processing_hour_str = ts_str(processing_hour)

    part = (
        source_df
        .filter(F.col("processing_date_hour") == F.lit(processing_hour_str))
        .orderBy(F.rand(random_state))
        .limit(int(max_rows))
        .withColumn("label", F.lit(float(label)))
    )

    return part


In [ ]:
positive_df = sample_by_processing_hour_spark(
    source_df=df_raw,
    processing_hour=target_processing_hour,
    label=1,
    max_rows=TARGET_SAMPLE_PER_GROUP,
    random_state=RANDOM_STATE
)

positive_count = positive_df.count()
print("Positive sample:")
print("Processing hour:", target_processing_hour)
print("Rows:", positive_count)

positive_df.select(
    "date_hour",
    "processing_date_hour",
    "label"
).show(5, truncate=False)

if positive_count < MIN_ROWS_PER_GROUP:
    raise ValueError(
        f"Positive sample quá ít: {positive_count} rows. "
        f"Cần tối thiểu {MIN_ROWS_PER_GROUP} rows."
    )

In [ ]:
negative_parts = []

for i, compare_hour in enumerate(comparison_processing_hours):
    part = sample_by_processing_hour_spark(
        source_df=df_raw,
        processing_hour=compare_hour,
        label=0,
        max_rows=TARGET_SAMPLE_PER_GROUP,
        random_state=RANDOM_STATE + i + 100
    )

    cnt = part.count()
    print("Comparison hour:", compare_hour, "| rows:", cnt)

    if cnt >= MIN_ROWS_PER_GROUP:
        negative_parts.append(part)
    else:
        print("Skipped because rows < MIN_ROWS_PER_GROUP")

if len(negative_parts) == 0:
    raise ValueError("Không có đủ dữ liệu đã có để tạo label = 0.")

negative_df = negative_parts[0]
for p in negative_parts[1:]:
    negative_df = negative_df.unionByName(p, allowMissingColumns=True)

negative_count = negative_df.count()
print("\nNegative sample total rows:", negative_count)

negative_df.groupBy("processing_date_hour") \
    .count() \
    .withColumnRenamed("count", "row_count") \
    .orderBy("processing_date_hour") \
    .show(truncate=False)

In [ ]:
work_df = positive_df.unionByName(negative_df, allowMissingColumns=True)

# Shuffle dữ liệu một lần, rồi persist để các cell sau không scan lại df_raw.
work_df = (
    work_df
    .orderBy(F.rand(RANDOM_STATE))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

row_count = work_df.count()
col_count = len(work_df.columns)

print("Work dataset shape:", (row_count, col_count))

print("\nLabel count:")
work_df.groupBy("label") \
    .count() \
    .orderBy("label") \
    .show(truncate=False)

print("\nRows by label and processing_date_hour:")
work_df.groupBy("label", "processing_date_hour") \
    .count() \
    .withColumnRenamed("count", "row_count") \
    .orderBy("label", "processing_date_hour") \
    .show(truncate=False)

print("\nSample rows:")
work_df.limit(5).show(truncate=False)

# Feature configuration


In [ ]:
EXCLUDE_COLS = [
    "label",
    "date_hour",
    "processing_date_hour",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
]

print("Columns to drop from model:")
for c in EXCLUDE_COLS:
    if c in work_df.columns:
        print("-", c)


In [ ]:
CATEGORICAL_COLS = [
    "VendorID",
    "RatecodeID",
    "store_and_fwd_flag",
    "payment_type",
]

NUMERIC_COLS = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "PULocationID",
    "DOLocationID",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee",
]

CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in work_df.columns]
NUMERIC_COLS = [c for c in NUMERIC_COLS if c in work_df.columns]

print("Categorical columns:", CATEGORICAL_COLS)
print("Numeric columns:", NUMERIC_COLS)


## Split and Spark feature pipeline


In [ ]:
def get_count_and_positive_rate(df, label_col="label"):
    row = df.select(
        F.count("*").alias("n"),
        F.avg(F.col(label_col).cast("double")).alias("positive_rate")
    ).first()
    return int(row["n"]), float(row["positive_rate"] or 0.0)


def stratified_split_exact_spark(df, label_col="label", test_size=0.15, valid_size=0.15, seed=42):
    """
    Spark split cố gắng khớp notebook LightGBM:
    - test = round(TEST_SIZE * n_total)
    - valid = round(VALID_SIZE * n_total)
    - stratify theo label.
    """
    counts = df.groupBy(label_col).count().collect()
    alloc_rows = []

    for r in counts:
        label_value = float(r[label_col])
        n = int(r["count"])
        n_test = int(round(test_size * n))
        n_valid = int(round(valid_size * n))
        alloc_rows.append((label_value, n_test, n_valid))

    alloc_df = spark.createDataFrame(alloc_rows, [label_col, "_n_test", "_n_valid"])

    w = Window.partitionBy(label_col).orderBy(F.rand(seed))

    ranked = (
        df.withColumn(label_col, F.col(label_col).cast("double"))
        .join(alloc_df, on=label_col, how="left")
        .withColumn("_rn", F.row_number().over(w))
        .withColumn(
            "_split",
            F.when(F.col("_rn") <= F.col("_n_test"), F.lit("test"))
             .when(F.col("_rn") <= F.col("_n_test") + F.col("_n_valid"), F.lit("valid"))
             .otherwise(F.lit("train"))
        )
        .drop("_n_test", "_n_valid", "_rn")
    )

    train_df = ranked.filter(F.col("_split") == "train").drop("_split")
    valid_df = ranked.filter(F.col("_split") == "valid").drop("_split")
    test_df = ranked.filter(F.col("_split") == "test").drop("_split")

    return train_df, valid_df, test_df


In [ ]:
class SparkTaxiFeaturePipeline:
    """
    Spark equivalent của TaxiFeatureEncoder:
    - Không tạo __isnull features.
    - Numeric giữ null/NaN ở vector.
    - Categorical string dùng StringIndexer(handleInvalid='keep') vì Spark vector cần numeric.
    - Categorical numeric cast double và khai báo categoricalSlotIndexes cho SynapseML.
    """
    def __init__(self, categorical_cols, numeric_cols, exclude_cols, label_col="label"):
        self.categorical_cols = list(categorical_cols)
        self.numeric_cols = list(numeric_cols)
        self.exclude_cols = set(exclude_cols)
        self.label_col = label_col

        self.numeric_feature_cols_ = []
        self.categorical_numeric_cols_ = []
        self.categorical_string_cols_ = []
        self.string_indexers_ = []
        self.string_indexed_cols_ = []
        self.feature_cols_ = []
        self.categorical_feature_cols_ = []
        self.categorical_slot_indexes_ = []
        self.assembler_ = None

    def fit(self, train_df):
        available = set(train_df.columns)

        self.numeric_feature_cols_ = [c for c in self.numeric_cols if c in available and c not in self.exclude_cols]

        # Phân tách categorical string và categorical numeric theo schema Spark
        schema_map = {f.name: f.dataType for f in train_df.schema.fields}
        self.categorical_string_cols_ = []
        self.categorical_numeric_cols_ = []

        for c in self.categorical_cols:
            if c not in available or c in self.exclude_cols:
                continue

            dtype = schema_map[c]
            if isinstance(dtype, T.StringType):
                self.categorical_string_cols_.append(c)
            else:
                self.categorical_numeric_cols_.append(c)

        df = train_df
        self.string_indexers_ = []
        self.string_indexed_cols_ = []

        for c in self.categorical_string_cols_:
            idx_col = f"{c}__idx"
            indexer = StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid="keep")
            model = indexer.fit(df)
            self.string_indexers_.append(model)
            self.string_indexed_cols_.append(idx_col)
            df = model.transform(df)

        self.feature_cols_ = self.numeric_feature_cols_ + self.categorical_numeric_cols_ + self.string_indexed_cols_
        self.categorical_feature_cols_ = self.categorical_numeric_cols_ + self.string_indexed_cols_
        self.categorical_slot_indexes_ = [self.feature_cols_.index(c) for c in self.categorical_feature_cols_]

        self.assembler_ = VectorAssembler(
            inputCols=self.feature_cols_,
            outputCol="features",
            handleInvalid="keep"
        )

        return self

    def transform(self, input_df):
        df = input_df

        # Cast numeric và categorical numeric sang double; inf -> null
        for c in self.numeric_feature_cols_ + self.categorical_numeric_cols_:
            if c in df.columns:
                df = df.withColumn(c, F.col(c).cast("double"))
                df = df.withColumn(
                    c,
                    F.when(F.isnan(F.col(c)) | F.col(c).isin(float("inf"), float("-inf")), F.lit(None).cast("double"))
                     .otherwise(F.col(c))
                )

        for model in self.string_indexers_:
            df = model.transform(df)

        df = self.assembler_.transform(df)

        return df


In [ ]:
train_raw_df, valid_raw_df, test_raw_df = stratified_split_exact_spark(
    work_df,
    label_col="label",
    test_size=TEST_SIZE,
    valid_size=VALID_SIZE,
    seed=RANDOM_STATE,
)

for name, part in [("Train raw", train_raw_df), ("Valid raw", valid_raw_df), ("Test raw", test_raw_df)]:
    n, pr = get_count_and_positive_rate(part)
    print(f"{name}: ({n}, {len(part.columns)}) | positive rate: {pr}")


In [ ]:
feature_pipeline = SparkTaxiFeaturePipeline(
    categorical_cols=CATEGORICAL_COLS,
    numeric_cols=NUMERIC_COLS,
    exclude_cols=EXCLUDE_COLS,
    label_col="label"
)

feature_pipeline.fit(train_raw_df)

# Transform một lần và persist.
# Đây là tối ưu rất quan trọng: Optuna không được build lại feature vector ở từng trial.
train_features_df = (
    feature_pipeline.transform(train_raw_df)
    .select("features", "label")
    .repartition(LGBM_NUM_PARTITIONS)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

valid_features_df = (
    feature_pipeline.transform(valid_raw_df)
    .select("features", "label")
    .repartition(LGBM_NUM_PARTITIONS)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

test_features_df = (
    feature_pipeline.transform(test_raw_df)
    .select("features", "label")
    .repartition(LGBM_NUM_PARTITIONS)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

print("Feature columns:")
print(feature_pipeline.feature_cols_)
print("Categorical feature columns:")
print(feature_pipeline.categorical_feature_cols_)
print("Categorical slot indexes:")
print(feature_pipeline.categorical_slot_indexes_)

print("X_train:", (train_features_df.count(), len(feature_pipeline.feature_cols_)))
print("X_valid:", (valid_features_df.count(), len(feature_pipeline.feature_cols_)))
print("X_test :", (test_features_df.count(), len(feature_pipeline.feature_cols_)))

In [ ]:
# Missing summary tương đương notebook LightGBM, nhưng tính trên raw feature columns trước VectorAssembler
missing_exprs = []
for c in feature_pipeline.numeric_feature_cols_ + feature_pipeline.categorical_numeric_cols_ + feature_pipeline.categorical_string_cols_:
    if c in train_raw_df.columns:
        missing_exprs.append(F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c))

if missing_exprs:
    missing_row = train_raw_df.select(*missing_exprs).first().asDict()
    missing_summary = pd.Series(missing_row).sort_values(ascending=False)
    display(missing_summary[missing_summary > 0])
else:
    print("No feature columns for missing summary.")


# SynapseML LightGBM helpers


In [ ]:
def build_lgbm_train_df(train_df, valid_df):
    return (
        train_df.withColumn("is_validation", F.lit(False))
        .unionByName(valid_df.withColumn("is_validation", F.lit(True)))
        .select("features", "label", "is_validation")
    )


# ============================================================
# SynapseML LightGBM params only
# ------------------------------------------------------------
# Notebook này chỉ dùng 1 kiểu params: tên tham số SynapseML.
# Các khoảng search space vẫn được ánh xạ tương đương bản LightGBM thường,
# nhưng KHÔNG chuyển đổi qua lại learning_rate/num_leaves/... nữa.
# ============================================================

SYNAPSEML_TUNABLE_PARAM_NAMES = {
    "learningRate",
    "numLeaves",
    "maxDepth",
    "minDataInLeaf",
    "baggingFraction",
    "baggingFreq",
    "featureFraction",
    "lambdaL1",
    "lambdaL2",
    "minGainToSplit",
    "maxBin",
}

SYNAPSEML_FIXED_PARAM_NAMES = {
    "numIterations",
    "earlyStoppingRound",
    "numTasks",
    "numThreads",
    "maxStreamingOMPThreads",
    "timeout",
    "useBarrierExecutionMode",
    "useSingleDatasetMode",
    "dataTransferMode",
    "verbosity",
    "driverListenPort",
    "defaultListenPort",
}


def with_fixed_synapseml_params(params):
    """
    Nhận params ĐÃ theo tên SynapseML và thêm fixed config để train.

    Không nhận / không map tên LightGBM thường như learning_rate, num_leaves,
    min_child_samples... nhằm tránh nhầm lẫn giữa 2 kiểu params.
    """
    params = dict(params)

    allowed = SYNAPSEML_TUNABLE_PARAM_NAMES | SYNAPSEML_FIXED_PARAM_NAMES
    unknown_keys = sorted(set(params) - allowed)
    if unknown_keys:
        raise ValueError(
            "Chỉ dùng tên tham số SynapseML trong notebook này. "
            f"Các key không hợp lệ: {unknown_keys}"
        )

    out = dict(params)

    # Fixed training config: không tune trong Optuna.
    out.update({
        "numIterations": int(out.get("numIterations", N_ESTIMATORS_MAX)),
        "earlyStoppingRound": int(out.get("earlyStoppingRound", EARLY_STOPPING_ROUNDS)),

        # Spark/SynapseML profile hiện tại: 6 tasks x 2 threads.
        "numTasks": int(out.get("numTasks", LGBM_NUM_TASKS)),
        "numThreads": int(out.get("numThreads", LGBM_NUM_THREADS)),
        "maxStreamingOMPThreads": int(out.get("maxStreamingOMPThreads", LGBM_NUM_THREADS)),

        "timeout": int(out.get("timeout", 600)),
        "useBarrierExecutionMode": bool(out.get("useBarrierExecutionMode", True)),
        "useSingleDatasetMode": bool(out.get("useSingleDatasetMode", True)),
        "dataTransferMode": out.get("dataTransferMode", "streaming"),
        "verbosity": int(out.get("verbosity", -1)),
    })

    if USE_FIXED_PORTS:
        out.update({
            "driverListenPort": int(out.get("driverListenPort", 12400)),
            "defaultListenPort": int(out.get("defaultListenPort", 12401)),
        })
    else:
        out.pop("driverListenPort", None)
        out.pop("defaultListenPort", None)

    return out


def build_synapseml_lgbm_classifier(
    synapseml_params,
    categorical_slot_indexes=None,
    shap_col=None,
    validation_col="is_validation",
):
    p = with_fixed_synapseml_params(synapseml_params)

    kwargs = {
        "objective": "binary",
        "metric": "auc",

        "featuresCol": "features",
        "labelCol": "label",
        "validationIndicatorCol": validation_col,

        # SynapseML / LightGBM params
        "learningRate": float(p.get("learningRate", 0.05)),
        "numLeaves": int(p.get("numLeaves", 31)),
        "maxDepth": int(p.get("maxDepth", -1)),
        "minDataInLeaf": int(p.get("minDataInLeaf", 20)),

        "baggingFraction": float(p.get("baggingFraction", 1.0)),
        "baggingFreq": int(p.get("baggingFreq", 0)),
        "featureFraction": float(p.get("featureFraction", 1.0)),

        "lambdaL1": float(p.get("lambdaL1", 0.0)),
        "lambdaL2": float(p.get("lambdaL2", 0.0)),
        "minGainToSplit": float(p.get("minGainToSplit", 0.0)),

        "maxBin": int(p.get("maxBin", 255)),
        "numIterations": int(p.get("numIterations", N_ESTIMATORS_MAX)),
        "earlyStoppingRound": int(p.get("earlyStoppingRound", EARLY_STOPPING_ROUNDS)),

        # Spark distributed LightGBM profile
        "numTasks": int(p.get("numTasks", LGBM_NUM_TASKS)),
        "numThreads": int(p.get("numThreads", LGBM_NUM_THREADS)),
        "maxStreamingOMPThreads": int(p.get("maxStreamingOMPThreads", LGBM_NUM_THREADS)),
        "timeout": int(p.get("timeout", 600)),

        "useBarrierExecutionMode": bool(p.get("useBarrierExecutionMode", True)),
        "useSingleDatasetMode": bool(p.get("useSingleDatasetMode", True)),
        "dataTransferMode": p.get("dataTransferMode", "streaming"),

        "verbosity": int(p.get("verbosity", -1)),
    }

    if USE_FIXED_PORTS:
        kwargs["driverListenPort"] = int(p.get("driverListenPort", 12400))
        kwargs["defaultListenPort"] = int(p.get("defaultListenPort", 12401))

    if categorical_slot_indexes:
        kwargs["categoricalSlotIndexes"] = list(categorical_slot_indexes)

    if shap_col:
        kwargs["featuresShapCol"] = shap_col

    return LightGBMClassifier(**kwargs)


def evaluate_auc(model, df, return_predictions=True):
    pred_df = model.transform(df)

    evaluator = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC",
    )

    auc = evaluator.evaluate(pred_df)

    if return_predictions:
        return float(auc), pred_df

    return float(auc), None


def extract_probability_label_df(pred_df, label_col="label", probability_col="probability"):
    score_df = (
        pred_df
        .select(
            F.col(label_col).cast("double").alias("y_true"),
            vector_to_array(probability_col)[1].alias("pred_score")
        )
    )
    return score_df.toPandas()


def safe_get_best_iteration(model):
    for attr in ["getBestIteration", "bestIteration"]:
        try:
            v = getattr(model, attr)
            return int(v() if callable(v) else v)
        except Exception:
            pass

    return None

# Baseline SynapseML LightGBM


In [ ]:
import time
from pyspark import StorageLevel

# ============================================================
# Baseline SynapseML LightGBM
# Mục tiêu: khớp baseline LightGBM thường về search/base params,
# nhưng giữ distributed config riêng cho Spark/SynapseML.
# ============================================================

baseline_search_params = {
    "learningRate": 0.05,
    "numLeaves": 31,
    "maxDepth": -1,
    "minDataInLeaf": 20,

    "baggingFraction": 1.0,
    "baggingFreq": 0,
    "featureFraction": 1.0,

    "lambdaL1": 1e-8,
    "lambdaL2": 1e-8,
    "minGainToSplit": 0.0,

    # Khớp LightGBM thường baseline max_bin=255.
    # Nếu executor native memory còn yếu, có thể đổi riêng baseline về 127.
    "maxBin": 255,
}

baseline_synapseml_params = with_fixed_synapseml_params(baseline_search_params)

baseline_lgbm = build_synapseml_lgbm_classifier(
    synapseml_params=baseline_synapseml_params,
    categorical_slot_indexes=feature_pipeline.categorical_slot_indexes_,
    shap_col=None,
    validation_col="is_validation",
)


# ============================================================
# Build train + validation dataframe cho LightGBM
# Dữ liệu này dùng lại cho baseline, Optuna và final model.
# Không unpersist cho tới khi tất cả các phần train chính đã xong.
# ============================================================

baseline_train_df = (
    build_lgbm_train_df(train_features_df, valid_features_df)
    .select("features", "label", "is_validation")
    .repartition(LGBM_NUM_PARTITIONS)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

valid_eval_df = (
    valid_features_df
    .select("features", "label")
    .repartition(LGBM_NUM_PARTITIONS)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

test_eval_df = (
    test_features_df
    .select("features", "label")
    .repartition(LGBM_NUM_PARTITIONS)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

# Materialize cache một lần
print("Baseline train + valid rows:", baseline_train_df.count())
print("Valid rows:", valid_eval_df.count())
print("Test rows :", test_eval_df.count())


# ============================================================
# Fit baseline model
# ============================================================

baseline_start = time.time()

baseline_model = baseline_lgbm.fit(baseline_train_df)
baseline_fit_elapsed = time.time() - baseline_start

baseline_valid_auc, baseline_valid_pred_df = evaluate_auc(
    baseline_model,
    valid_eval_df
)

baseline_test_auc, baseline_test_pred_df = evaluate_auc(
    baseline_model,
    test_eval_df
)

baseline_best_iteration = safe_get_best_iteration(baseline_model)

print("Baseline valid AUC       :", baseline_valid_auc)
print("Baseline test AUC        :", baseline_test_auc)
print("Baseline best iteration  :", baseline_best_iteration)
print("Baseline fit seconds     :", baseline_fit_elapsed)

# Optuna tuning


In [ ]:
import time
import gc

def suggest_lgbm_params(trial):
    # Search space giữ tương đương notebook LightGBM thường,
    # chỉ đổi tên tham số sang SynapseML.
    # numIterations cố định bằng N_ESTIMATORS_MAX, không đưa vào search space.

    max_depth = trial.suggest_categorical(
        "maxDepth",
        [-1, 4, 5, 6, 7, 8, 10, 12]
    )

    if max_depth == -1:
        num_leaves_upper = 256
    else:
        num_leaves_upper = min(256, 2 ** max_depth)

    params = {
        # 1. Learning speed
        "learningRate": trial.suggest_float(
            "learningRate",
            0.01,
            0.15,
            log=True
        ),

        # 2. Tree complexity
        "numLeaves": trial.suggest_int(
            "numLeaves",
            16,
            num_leaves_upper
        ),
        "maxDepth": max_depth,

        # 3. Prevent overfitting in leaf-wise trees
        # Giữ giống LightGBM thường: 10 -> 1000 log
        "minDataInLeaf": trial.suggest_int(
            "minDataInLeaf",
            10,
            1000,
            log=True
        ),

        # 4. Row bagging
        # Giữ giống LightGBM thường: baggingFreq vẫn search 0 -> 10.
        # Nếu baggingFreq = 0 thì baggingFraction không có hiệu lực, đúng như bản thường.
        "baggingFraction": trial.suggest_float(
            "baggingFraction",
            0.7,
            1.0
        ),
        "baggingFreq": trial.suggest_int(
            "baggingFreq",
            0,
            10
        ),

        # 5. Feature sampling
        "featureFraction": trial.suggest_float(
            "featureFraction",
            0.7,
            1.0
        ),

        # 6. Regularization
        "lambdaL1": trial.suggest_float(
            "lambdaL1",
            1e-8,
            10.0,
            log=True
        ),
        "lambdaL2": trial.suggest_float(
            "lambdaL2",
            1e-8,
            10.0,
            log=True
        ),

        # 7. Split control
        "minGainToSplit": trial.suggest_float(
            "minGainToSplit",
            0.0,
            1.0
        ),

        # 8. Histogram binning
        # Giữ giống LightGBM thường.
        "maxBin": trial.suggest_categorical(
            "maxBin",
            [63, 127, 255]
        ),
    }

    return with_fixed_synapseml_params(params)


def objective(trial):
    synapseml_params = suggest_lgbm_params(trial)

    model_estimator = build_synapseml_lgbm_classifier(
        synapseml_params=synapseml_params,
        categorical_slot_indexes=feature_pipeline.categorical_slot_indexes_,
        shap_col=None,
        validation_col="is_validation",
    )

    trial_start = time.time()

    fit_start = time.time()
    model = model_estimator.fit(baseline_train_df)
    fit_elapsed = time.time() - fit_start

    eval_start = time.time()
    valid_auc, _ = evaluate_auc(
        model,
        valid_eval_df,
        return_predictions=False
    )
    eval_elapsed = time.time() - eval_start

    total_elapsed = time.time() - trial_start

    trial.set_user_attr("valid_auc", float(valid_auc))
    trial.set_user_attr("fit_seconds", float(fit_elapsed))
    trial.set_user_attr("eval_seconds", float(eval_elapsed))
    trial.set_user_attr("elapsed_seconds", float(total_elapsed))

    best_iter = safe_get_best_iteration(model)
    if best_iter is not None:
        trial.set_user_attr("best_iteration", int(best_iter))

    print(
        f"Trial {trial.number} | "
        f"AUC={valid_auc:.6f} | "
        f"fit={fit_elapsed:.1f}s | "
        f"eval={eval_elapsed:.1f}s | "
        f"total={total_elapsed:.1f}s | "
        f"best_iter={best_iter}"
    )

    del model
    gc.collect()

    return float(valid_auc)

In [ ]:
sampler = optuna.samplers.TPESampler(
    seed=RANDOM_STATE,
    n_startup_trials=10
)

# Khác LightGBM thường: SynapseML không expose AUC từng boosting round ra Python callback.
# Vì vậy MedianPruner không cắt được trial sớm trong objective hiện tại.
# Early stopping vẫn được xử lý bên trong LightGBM bằng earlyStoppingRound.
pruner = optuna.pruners.NopPruner()

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    study_name="synapseml_lightgbm_auc_tuning"
)

print("Optuna study created.")
print("Direction:", study.direction)
print("Sampler:", type(study.sampler).__name__)
print("Pruner:", type(study.pruner).__name__)
print("Fixed numIterations:", N_ESTIMATORS_MAX)
print("Fixed earlyStoppingRound:", EARLY_STOPPING_ROUNDS)
print("Search space: matched with LightGBM notebook, SynapseML parameter names.")

In [ ]:
study.enqueue_trial({
    "learningRate": 0.05,
    "numLeaves": 31,
    "maxDepth": -1,
    "minDataInLeaf": 20,

    "baggingFraction": 1.0,
    "baggingFreq": 0,
    "featureFraction": 1.0,

    "lambdaL1": 1e-8,
    "lambdaL2": 1e-8,
    "minGainToSplit": 0.0,

    # Khớp LightGBM thường baseline max_bin=255.
    "maxBin": 255,
})

if RUN_OPTUNA:
    study.optimize(
        objective,
        n_trials=N_TRIALS,
        n_jobs=1,
        show_progress_bar=True,
        gc_after_trial=True
    )
else:
    # Chỉ chạy baseline trial đã enqueue ở trên
    study.optimize(
        objective,
        n_trials=1,
        n_jobs=1,
        show_progress_bar=True,
        gc_after_trial=True
    )

print("Best validation AUC:", study.best_value)
print("Best params:")
print(study.best_params)

print("Best trial attrs:")
print(study.best_trial.user_attrs)

In [ ]:
# Baseline search params theo kiểu SynapseML, khớp baseline LightGBM thường.
baseline_search_params = {
    "learningRate": 0.05,
    "numLeaves": 31,
    "maxDepth": -1,
    "minDataInLeaf": 20,
    "baggingFraction": 1.0,
    "baggingFreq": 0,
    "featureFraction": 1.0,
    "lambdaL1": 1e-8,
    "lambdaL2": 1e-8,
    "minGainToSplit": 0.0,
    "maxBin": 255,
}

if study.best_value >= baseline_valid_auc:
    selected_params = dict(study.best_params)
    selected_param_source = "optuna"
    selected_valid_auc = study.best_value
else:
    selected_params = dict(baseline_search_params)
    selected_param_source = "baseline"
    selected_valid_auc = baseline_valid_auc

# Notebook này chỉ dùng 1 kiểu params: SynapseML-style.
selected_synapseml_params = with_fixed_synapseml_params(selected_params)

print("Selected param source:", selected_param_source)
print("Selected validation AUC:", selected_valid_auc)
print("Selected SynapseML params:")
print(selected_params)
print("Selected full SynapseML train params:")
print(selected_synapseml_params)

# Final SynapseML model


In [ ]:
final_shap_col = "featuresShap" if RUN_FINAL_SHAP else None

final_lgbm = build_synapseml_lgbm_classifier(
    selected_synapseml_params,
    categorical_slot_indexes=feature_pipeline.categorical_slot_indexes_,
    shap_col=final_shap_col,
    validation_col="is_validation",
)

final_start = time.time()
final_model = final_lgbm.fit(baseline_train_df)
final_elapsed = time.time() - final_start

final_valid_auc, final_valid_pred_df = evaluate_auc(final_model, valid_eval_df)
final_test_auc, final_test_pred_df = evaluate_auc(final_model, test_eval_df)
final_best_iteration = safe_get_best_iteration(final_model)

print("Selected param source:", selected_param_source)
print("Final valid AUC:", final_valid_auc)
print("Final test AUC :", final_test_auc)
print("Final best iteration:", final_best_iteration)
print("Final elapsed seconds:", final_elapsed)
print("RUN_FINAL_SHAP:", RUN_FINAL_SHAP)

In [ ]:
auc_summary_df = pd.DataFrame([
    {
        "case": "baseline",
        "valid_auc": float(baseline_valid_auc),
        "test_auc": float(baseline_test_auc),
        "param_source": "baseline"
    },
    {
        "case": "optuna_best",
        "valid_auc": float(study.best_value),
        "test_auc": None,
        "param_source": "optuna"
    },
    {
        "case": "final_selected",
        "valid_auc": float(final_valid_auc),
        "test_auc": float(final_test_auc),
        "param_source": selected_param_source
    },
])

display(auc_summary_df)


In [ ]:
trials_df = study.trials_dataframe()

display(
    trials_df.sort_values("value", ascending=False).head(10)
)

trials_path = f"{OUTPUT_DIR}/optuna_synapseml_lgbm_trials.csv"
trials_df.to_csv(trials_path, index=False)

print("Saved:", trials_path)


In [ ]:
from optuna.importance import get_param_importances

try:
    param_importance = get_param_importances(study)

    param_importance_df = pd.DataFrame({
        "parameter": list(param_importance.keys()),
        "importance": list(param_importance.values())
    }).sort_values("importance", ascending=False)

    display(param_importance_df)

    param_importance_path = f"{OUTPUT_DIR}/optuna_synapseml_param_importance.csv"
    param_importance_df.to_csv(param_importance_path, index=False)

    print("Saved:", param_importance_path)
except Exception as e:
    print("Param importance skipped:", repr(e))


In [ ]:
best_config = {
    "model": "SynapseML LightGBM",
    "task": "tlc_yellow_taxi_current_batch_vs_existing_batches",
    "target_processing_date_hour": str(TARGET_PROCESSING_DATE_HOUR),
    "comparison_processing_date_hours": [str(h) for h in comparison_processing_hours],
    "metric": "auc",

    "baseline_valid_auc": float(baseline_valid_auc),
    "baseline_test_auc": float(baseline_test_auc),

    "optuna_best_valid_auc": float(study.best_value),
    "optuna_best_params_synapseml": study.best_params,

    "selected_param_source": selected_param_source,
    "selected_valid_auc": float(selected_valid_auc),
    "selected_params_synapseml": selected_params,
    "selected_full_synapseml_train_params": selected_synapseml_params,

    "final_valid_auc": float(final_valid_auc),
    "final_test_auc": float(final_test_auc),
    "final_best_iteration": None if final_best_iteration is None else int(final_best_iteration),

    "feature_cols": feature_pipeline.feature_cols_,
    "categorical_feature_cols": feature_pipeline.categorical_feature_cols_,
    "categorical_slot_indexes": feature_pipeline.categorical_slot_indexes_,

    "fixed_training_config": {
        "n_estimators_max": N_ESTIMATORS_MAX,
        "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
        "random_state": RANDOM_STATE,
        "target_sample_per_group": TARGET_SAMPLE_PER_GROUP,
        "min_rows_per_group": MIN_ROWS_PER_GROUP,
        "previous_batch_offset": str(PREVIOUS_BATCH_OFFSET),
        "history_offsets": [str(x) for x in HISTORY_OFFSETS],
        "no_isnull_features": True,
        "no_trip_duration_min": True,
        "spark_engine": "synapseml_lgbm",
        "lgbm_num_tasks": LGBM_NUM_TASKS,
        "lgbm_num_threads": LGBM_NUM_THREADS,
        "lgbm_num_partitions": LGBM_NUM_PARTITIONS,
        "run_final_shap": RUN_FINAL_SHAP,
    },

    "tuned_parameter_groups": {
        "tree_complexity": [
            "numLeaves",
            "maxDepth",
            "minDataInLeaf"
        ],
        "learning_process": [
            "learningRate",
            "numIterations_with_early_stopping"
        ],
        "sampling": [
            "baggingFraction",
            "baggingFreq",
            "featureFraction"
        ],
        "regularization": [
            "lambdaL1",
            "lambdaL2",
            "minGainToSplit"
        ],
        "binning": [
            "maxBin"
        ]
    }
}

best_params_path = f"{OUTPUT_DIR}/best_synapseml_lgbm_params.json"

with open(best_params_path, "w") as f:
    json.dump(best_config, f, indent=2, default=str)

print("Saved:", best_params_path)
display(best_config)

# Classification report style


In [ ]:
threshold = 0.2
final_test_score_pdf = extract_probability_label_df(final_test_pred_df)
final_test_label_pred = (final_test_score_pdf["pred_score"].values >= threshold).astype(int)

print(classification_report(final_test_score_pdf["y_true"].values, final_test_label_pred))

display(
    final_test_score_pdf.groupby("y_true")["pred_score"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)


# Feature importance and SHAP


In [ ]:
def get_synapseml_feature_importance_df(model, feature_cols):
    importances = None

    # SynapseML versions expose feature importance differently. Try common APIs.
    for expr in [
        "model.getFeatureImportances()",
        "model.getLightGBMBooster().getFeatureImportances()",
    ]:
        try:
            importances = eval(expr)
            break
        except Exception:
            pass

    if importances is None:
        print("Could not read native feature_importances from SynapseML model. Use SHAP top features below instead.")
        return pd.DataFrame({"feature": feature_cols, "importance": np.nan})

    importances = list(importances)
    if len(importances) != len(feature_cols):
        print("Warning: importance length does not match feature_cols length", len(importances), len(feature_cols))

    n = min(len(importances), len(feature_cols))
    return pd.DataFrame({
        "feature": feature_cols[:n],
        "importance": importances[:n]
    }).sort_values("importance", ascending=False)

importance_df = get_synapseml_feature_importance_df(final_model, feature_pipeline.feature_cols_)
display(importance_df.head(30))

importance_path = f"{OUTPUT_DIR}/final_synapseml_feature_importance.csv"
importance_df.to_csv(importance_path, index=False)
print("Saved:", importance_path)


In [ ]:
import matplotlib.pyplot as plt

top_n = 20
plot_df = importance_df.dropna().head(top_n).sort_values("importance")

if len(plot_df) > 0:
    plt.figure(figsize=(8, 6))
    plt.barh(plot_df["feature"], plot_df["importance"])
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.title("Top SynapseML LightGBM Feature Importance")
    plt.show()
else:
    print("Native feature importance not available; use SHAP plot below.")


In [ ]:
def compute_top_shap_from_predictions(pred_df, feature_cols, shap_col="featuresShap", top_n=20, sample_size=None, seed=42):
    if shap_col not in pred_df.columns:
        raise ValueError(
            f"Column {shap_col!r} không tồn tại trong prediction DataFrame. "
            "Hãy bật RUN_FINAL_SHAP=True hoặc train estimator với featuresShapCol."
        )

    shap_source_df = pred_df
    if sample_size is not None:
        total = pred_df.count()
        if total > sample_size:
            frac = min(1.0, float(sample_size) / float(total) * 1.5)
            shap_source_df = pred_df.sample(False, frac, seed=seed).limit(int(sample_size))

    shap_df = shap_source_df.withColumn("shap_array", vector_to_array(shap_col))
    shap_size = shap_df.select(F.size("shap_array").alias("shap_size")).first()["shap_size"]

    shap_long_df = shap_df.select(
        F.posexplode("shap_array").alias("feature_index", "shap_value")
    )

    # Nếu có bias/base value cuối vector, bỏ phần cuối.
    if shap_size == len(feature_cols) + 1:
        shap_long_df = shap_long_df.filter(F.col("feature_index") < len(feature_cols))

    feature_mapping_df = spark.createDataFrame(
        [(i, name) for i, name in enumerate(feature_cols)],
        ["feature_index", "feature"]
    )

    shap_all_df = (
        shap_long_df
        .groupBy("feature_index")
        .agg(F.avg(F.abs("shap_value")).alias("mean_abs_shap"))
        .join(feature_mapping_df, on="feature_index", how="left")
        .orderBy(F.desc("mean_abs_shap"))
    )

    return shap_all_df.limit(top_n), shap_all_df


if RUN_FINAL_SHAP and "featuresShap" in final_test_pred_df.columns:
    SHAP_SAMPLE_SIZE = min(3000, test_eval_df.count())
    shap_top_df_spark, shap_all_df_spark = compute_top_shap_from_predictions(
        final_test_pred_df,
        feature_cols=feature_pipeline.feature_cols_,
        shap_col="featuresShap",
        top_n=20,
        sample_size=SHAP_SAMPLE_SIZE,
        seed=RANDOM_STATE
    )

    shap_top_pdf = shap_top_df_spark.toPandas()
    display(shap_top_pdf)

    shap_path = f"{OUTPUT_DIR}/final_synapseml_shap_top_features.csv"
    shap_top_pdf.to_csv(shap_path, index=False)
    print("Saved:", shap_path)
else:
    shap_top_pdf = pd.DataFrame(columns=["feature_index", "mean_abs_shap", "feature"])
    print("Final SHAP skipped. Set RUN_FINAL_SHAP=True to compute SynapseML featuresShap.")

In [ ]:
if len(shap_top_pdf) > 0:
    plot_df = shap_top_pdf.sort_values("mean_abs_shap")

    plt.figure(figsize=(8, 6))
    plt.barh(plot_df["feature"], plot_df["mean_abs_shap"])
    plt.xlabel("Mean absolute SHAP value")
    plt.ylabel("Feature")
    plt.title("Top SynapseML SHAP Features")
    plt.show()
else:
    print("No SHAP plot because SHAP was skipped or unavailable.")

# Synthetic anomaly injection


In [ ]:
# Tỷ lệ bản ghi label = 1 bị giả lập bất thường
# Giữ cùng mục tiêu với notebook LightGBM, nhưng chỉ chạy khi RUN_SYNTHETIC_EXPERIMENTS=True
# để tránh tốn thêm Spark actions trong fast path.
ANOMALY_FRACTION = 0.30
ANOMALY_RANDOM_STATE = 2025

print("Synthetic anomaly fraction:", ANOMALY_FRACTION)

if RUN_SYNTHETIC_EXPERIMENTS:
    print("Positive target rows:", positive_df.count())
    print("Negative history rows:", negative_df.count())

    print("Target processing hour:")
    display(positive_df.groupBy("processing_date_hour").count().toPandas())

    print("History processing hours:")
    display(negative_df.groupBy("processing_date_hour").count().orderBy("processing_date_hour").toPandas())
else:
    print("Synthetic experiment summary skipped.")

In [ ]:
def inject_synthetic_anomalies_spark(target_df, anomaly_fraction=0.30, random_state=42):
    """
    Spark version của inject_synthetic_anomalies trong notebook LightGBM.
    Giữ cùng ý tưởng 5 nhóm anomaly:
    1. fare/tip/total tăng bất thường
    2. trip_distance rất lớn
    3. passenger_count bất thường hoặc missing
    4. category lạ payment_type/store_and_fwd_flag
    5. location ID bất thường
    """
    data = (
        target_df
        .withColumn("_r_anom", F.rand(random_state))
        .withColumn("synthetic_anomaly", F.when(F.col("_r_anom") < F.lit(float(anomaly_fraction)), F.lit(1)).otherwise(F.lit(0)))
        .withColumn("_anom_group", F.floor(F.rand(random_state + 1) * F.lit(5)).cast("int"))
        .withColumn("_r1", F.rand(random_state + 2))
        .withColumn("_r2", F.rand(random_state + 3))
        .withColumn("_r3", F.rand(random_state + 4))
    )

    is_anom = F.col("synthetic_anomaly") == 1

    # 1. Fare / total amount tăng bất thường
    if "fare_amount" in data.columns:
        data = data.withColumn(
            "fare_amount",
            F.when(is_anom & (F.col("_anom_group") == 0), F.col("fare_amount").cast("double") * (F.lit(3.0) + F.col("_r1") * F.lit(5.0)))
             .otherwise(F.col("fare_amount"))
        )

    if "tip_amount" in data.columns:
        data = data.withColumn(
            "tip_amount",
            F.when(is_anom & (F.col("_anom_group") == 0), F.lit(20.0) + F.col("_r2") * F.lit(80.0))
             .otherwise(F.col("tip_amount"))
        )

    if "total_amount" in data.columns:
        data = data.withColumn(
            "total_amount",
            F.when(is_anom & (F.col("_anom_group") == 0), F.col("fare_amount").cast("double") + F.lit(50.0) + F.col("_r3") * F.lit(100.0))
             .otherwise(F.col("total_amount"))
        )

    # 2. Trip distance rất lớn
    if "trip_distance" in data.columns:
        data = data.withColumn(
            "trip_distance",
            F.when(is_anom & (F.col("_anom_group") == 1), F.lit(80.0) + F.col("_r1") * F.lit(100.0))
             .otherwise(F.col("trip_distance"))
        )

    if "total_amount" in data.columns:
        data = data.withColumn(
            "total_amount",
            F.when(is_anom & (F.col("_anom_group") == 1), F.lit(200.0) + F.col("_r2") * F.lit(300.0))
             .otherwise(F.col("total_amount"))
        )

    # 3. Passenger count bất thường hoặc missing
    if "passenger_count" in data.columns:
        data = data.withColumn(
            "passenger_count",
            F.when(is_anom & (F.col("_anom_group") == 2) & (F.col("_r1") < 0.5), F.floor(F.lit(8) + F.col("_r2") * F.lit(7)).cast("double"))
             .when(is_anom & (F.col("_anom_group") == 2), F.lit(None).cast("double"))
             .otherwise(F.col("passenger_count"))
        )

    # 4. Category lạ
    if "payment_type" in data.columns:
        data = data.withColumn(
            "payment_type",
            F.when(is_anom & (F.col("_anom_group") == 3), F.lit(99))
             .otherwise(F.col("payment_type"))
        )

    if "store_and_fwd_flag" in data.columns:
        data = data.withColumn(
            "store_and_fwd_flag",
            F.when(is_anom & (F.col("_anom_group") == 3), F.lit("ANOM"))
             .otherwise(F.col("store_and_fwd_flag"))
        )

    # 5. Location ID bất thường
    for loc_col in ["PULocationID", "DOLocationID"]:
        if loc_col in data.columns:
            data = data.withColumn(
                loc_col,
                F.when(is_anom & (F.col("_anom_group") == 4), F.lit(999))
                 .otherwise(F.col(loc_col))
            )

    return data.drop("_r_anom", "_anom_group", "_r1", "_r2", "_r3")


In [ ]:
if RUN_SYNTHETIC_EXPERIMENTS:
    abnormal_positive_df = inject_synthetic_anomalies_spark(
        target_df=positive_df,
        anomaly_fraction=ANOMALY_FRACTION,
        random_state=ANOMALY_RANDOM_STATE
    ).persist(StorageLevel.MEMORY_AND_DISK)

    print("Abnormal positive shape:", (abnormal_positive_df.count(), len(abnormal_positive_df.columns)))
    print("Synthetic anomaly counts:")
    display(abnormal_positive_df.groupBy("synthetic_anomaly").count().orderBy("synthetic_anomaly").toPandas())

    display(
        abnormal_positive_df.select(
            "processing_date_hour",
            "label",
            "synthetic_anomaly",
            *[c for c in ["trip_distance", "fare_amount", "tip_amount", "total_amount", "passenger_count", "payment_type", "PULocationID", "DOLocationID"] if c in abnormal_positive_df.columns]
        ).limit(20).toPandas()
    )
else:
    print("Synthetic anomaly construction skipped.")

In [ ]:
if RUN_SYNTHETIC_EXPERIMENTS:
    normal_negative_df = negative_df.withColumn("synthetic_anomaly", F.lit(0))

    abnormal_work_df = abnormal_positive_df.unionByName(normal_negative_df, allowMissingColumns=True)
    abnormal_work_df = (
        abnormal_work_df
        .orderBy(F.rand(RANDOM_STATE))
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    print("Abnormal work dataset shape:", (abnormal_work_df.count(), len(abnormal_work_df.columns)))
    display(abnormal_work_df.groupBy("label").count().orderBy("label").toPandas())
    display(abnormal_work_df.groupBy("label", "synthetic_anomaly").count().orderBy("label", "synthetic_anomaly").toPandas())
else:
    print("Synthetic work dataset skipped.")

In [ ]:
if RUN_SYNTHETIC_EXPERIMENTS:
    compare_cols = [
        "trip_distance",
        "fare_amount",
        "tip_amount",
        "total_amount",
        "passenger_count",
    ]
    compare_cols = [c for c in compare_cols if c in abnormal_work_df.columns]

    summary_rows = []
    for name, df_part in [
        ("normal_target", positive_df),
        ("abnormal_target", abnormal_positive_df),
        ("history", negative_df),
    ]:
        agg_exprs = []
        for c in compare_cols:
            agg_exprs += [
                F.mean(F.col(c).cast("double")).alias(f"{c}__mean"),
                F.expr(f"percentile_approx(CAST({c} AS DOUBLE), 0.5)").alias(f"{c}__50%"),
                F.max(F.col(c).cast("double")).alias(f"{c}__max"),
            ]
        row = df_part.select(*agg_exprs).first().asDict()
        for c in compare_cols:
            summary_rows.append({
                "case": name,
                "column": c,
                "mean": row.get(f"{c}__mean"),
                "50%": row.get(f"{c}__50%"),
                "max": row.get(f"{c}__max"),
            })

    summary_stats = pd.DataFrame(summary_rows)
    display(summary_stats)
else:
    print("Synthetic summary skipped.")

In [ ]:
from pyspark import StorageLevel
from pyspark.sql import functions as F


def train_synapseml_on_work_df(
    input_work_df,
    synapseml_params,
    random_state=RANDOM_STATE,
    shap_col=None,
    return_artifacts=True,
):
    """
    Train/evaluate một dataset Spark bằng SynapseML LightGBM.

    Tối ưu:
    - Không gọi spark.catalog.clearCache() trong function để tránh xóa cache global của baseline/Optuna.
    - Chỉ persist các DataFrame cần dùng trong function.
    - Nếu return_artifacts=False, tự unpersist train/valid/test sau khi evaluate để tránh rò cache khi chạy batch/sweep.
    """
    train_raw, valid_raw, test_raw = stratified_split_exact_spark(
        input_work_df,
        label_col="label",
        test_size=TEST_SIZE,
        valid_size=VALID_SIZE,
        seed=random_state,
    )

    pipeline = SparkTaxiFeaturePipeline(
        categorical_cols=CATEGORICAL_COLS,
        numeric_cols=NUMERIC_COLS,
        exclude_cols=EXCLUDE_COLS + ["synthetic_anomaly"],
        label_col="label",
    )

    pipeline.fit(train_raw)

    tr = (
        pipeline.transform(train_raw)
        .select("features", "label")
        .repartition(LGBM_NUM_PARTITIONS)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    va = (
        pipeline.transform(valid_raw)
        .select("features", "label")
        .repartition(LGBM_NUM_PARTITIONS)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    te = (
        pipeline.transform(test_raw)
        .select("features", "label")
        .repartition(LGBM_NUM_PARTITIONS)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    n_train = tr.count()
    n_valid = va.count()
    n_test = te.count()

    syn_params = with_fixed_synapseml_params(synapseml_params)

    estimator = build_synapseml_lgbm_classifier(
        synapseml_params=syn_params,
        categorical_slot_indexes=pipeline.categorical_slot_indexes_,
        shap_col=shap_col,
        validation_col="is_validation",
    )

    train_valid = (
        build_lgbm_train_df(tr, va)
        .repartition(LGBM_NUM_PARTITIONS)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    train_valid.count()

    try:
        model = estimator.fit(train_valid)

        valid_auc, valid_pred = evaluate_auc(model, va)
        test_auc, test_pred = evaluate_auc(model, te)

        result = {
            "valid_auc": float(valid_auc),
            "test_auc": float(test_auc),
            "best_iteration": safe_get_best_iteration(model),
            "n_train": int(n_train),
            "n_valid": int(n_valid),
            "n_test": int(n_test),
            "feature_cols": pipeline.feature_cols_,
            "categorical_slot_indexes": pipeline.categorical_slot_indexes_,
        }

        artifacts = {
            "model": model,
            "pipeline": pipeline,
            "train_features_df": tr,
            "valid_features_df": va,
            "test_features_df": te,
            "test_raw_df": test_raw,
            "test_pred_df": test_pred,
        }

        if return_artifacts:
            return result, artifacts

        return result, None

    finally:
        train_valid.unpersist()

        if not return_artifacts:
            tr.unpersist()
            va.unpersist()
            te.unpersist()

In [ ]:
if RUN_SYNTHETIC_EXPERIMENTS:
    abnormal_result, abnormal_artifacts = train_synapseml_on_work_df(
        abnormal_work_df,
        synapseml_params=selected_params,
        random_state=RANDOM_STATE,
        shap_col=("featuresShap" if RUN_FINAL_SHAP else None),
        return_artifacts=True,
    )

    abnormal_valid_auc = abnormal_result["valid_auc"]
    abnormal_test_auc = abnormal_result["test_auc"]

    print("Abnormal valid AUC:", abnormal_valid_auc)
    print("Abnormal test AUC :", abnormal_test_auc)
    print("Best iteration:", abnormal_result["best_iteration"])
else:
    print("Synthetic experiments are disabled.")

In [ ]:
if RUN_SYNTHETIC_EXPERIMENTS:
    comparison_rows = []

    if "baseline_test_auc" in globals():
        comparison_rows.append({
            "case": "Normal data - baseline SynapseML LightGBM",
            "test_auc": float(baseline_test_auc)
        })

    if "final_test_auc" in globals():
        comparison_rows.append({
            "case": "Normal data - tuned SynapseML LightGBM",
            "test_auc": float(final_test_auc)
        })

    comparison_rows.append({
        "case": f"Synthetic anomaly data - tuned SynapseML LightGBM, anomaly_fraction={ANOMALY_FRACTION}",
        "test_auc": float(abnormal_test_auc)
    })

    auc_comparison_df = pd.DataFrame(comparison_rows)
    display(auc_comparison_df)


In [ ]:
if RUN_SYNTHETIC_EXPERIMENTS:
    ab_score_pdf = extract_probability_label_df(abnormal_artifacts["test_pred_df"])
    display(
        ab_score_pdf.groupby("y_true")["pred_score"]
        .agg(["count", "mean", "median", "min", "max"])
        .reset_index()
    )

    if "featuresShap" in abnormal_artifacts["test_pred_df"].columns:
        ab_shap_top_df, ab_shap_all_df = compute_top_shap_from_predictions(
            abnormal_artifacts["test_pred_df"],
            feature_cols=abnormal_result["feature_cols"],
            shap_col="featuresShap",
            top_n=20,
            sample_size=None,
            seed=RANDOM_STATE
        )
        ab_shap_top_pdf = ab_shap_top_df.toPandas()
        display(ab_shap_top_pdf)

        ab_shap_path = f"{OUTPUT_DIR}/abnormal_synapseml_shap_top_features.csv"
        ab_shap_top_pdf.to_csv(ab_shap_path, index=False)
        print("Saved:", ab_shap_path)
    else:
        print("Synthetic SHAP skipped. Enable RUN_FINAL_SHAP=True before training abnormal model.")

In [ ]:
def run_synthetic_anomaly_experiment(anomaly_fraction, random_state=42):
    temp_positive = inject_synthetic_anomalies_spark(
        target_df=positive_df,
        anomaly_fraction=anomaly_fraction,
        random_state=random_state
    )

    temp_negative = negative_df.withColumn("synthetic_anomaly", F.lit(0))

    temp_work_df = temp_positive.unionByName(temp_negative, allowMissingColumns=True)
    temp_work_df = (
        temp_work_df
        .orderBy(F.rand(random_state))
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    temp_result, _ = train_synapseml_on_work_df(
        temp_work_df,
        synapseml_params=selected_params,
        random_state=random_state,
        shap_col=None,
        return_artifacts=False,
    )

    result = {
        "anomaly_fraction": anomaly_fraction,
        "valid_auc": float(temp_result["valid_auc"]),
        "test_auc": float(temp_result["test_auc"]),
        "best_iteration": temp_result["best_iteration"],
        "n_rows": int(temp_work_df.count()),
        "n_positive": int(temp_work_df.filter(F.col("label") == 1).count()),
        "n_negative": int(temp_work_df.filter(F.col("label") == 0).count()),
        "n_synthetic_anomaly": int(temp_work_df.filter(F.col("synthetic_anomaly") == 1).count()),
    }

    temp_work_df.unpersist()
    return result

In [ ]:
ANOMALY_LEVELS = [0.0, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1]

sweep_results = []

if RUN_SYNTHETIC_EXPERIMENTS:
    for frac in ANOMALY_LEVELS:
        print("Running anomaly_fraction =", frac)

        result = run_synthetic_anomaly_experiment(
            anomaly_fraction=frac,
            random_state=ANOMALY_RANDOM_STATE
        )

        sweep_results.append(result)

    sweep_df = pd.DataFrame(sweep_results)
    display(sweep_df)
else:
    sweep_df = pd.DataFrame()
    print("Synthetic sweep is disabled.")


In [ ]:
if len(sweep_df) > 0:
    plt.figure(figsize=(7, 5))
    plt.plot(
        sweep_df["anomaly_fraction"],
        sweep_df["test_auc"],
        marker="o"
    )
    plt.xlabel("Synthetic anomaly fraction")
    plt.ylabel("Test AUC")
    plt.title("AUC vs Synthetic Anomaly Fraction")
    plt.grid(True)
    plt.show()


# Batch AUC monitoring


In [ ]:
BATCH_START = pd.Timestamp("2025-01-20 00:00:00")
BATCH_END = pd.Timestamp("2025-01-31 12:00:00")

TARGET_BATCH_LIST = pd.date_range(
    start=BATCH_START,
    end=BATCH_END,
    freq="12h"
).tolist()

print("Number of target batches:", len(TARGET_BATCH_LIST))
print("First target batch:", TARGET_BATCH_LIST[0])
print("Last target batch :", TARGET_BATCH_LIST[-1])

TARGET_BATCH_LIST[:5], TARGET_BATCH_LIST[-5:]


In [ ]:
if "selected_params" in globals():
    MODEL_PARAMS = selected_params
    PARAM_SOURCE = "selected_params"
elif "study" in globals():
    MODEL_PARAMS = study.best_params
    PARAM_SOURCE = "study.best_params"
else:
    raise ValueError("Không tìm thấy selected_params hoặc study.best_params.")

MODEL_PARAMS = dict(MODEL_PARAMS)

print("Using params from:", PARAM_SOURCE)
print(MODEL_PARAMS)

In [ ]:
def get_comparison_processing_hours(target_processing_hour):
    comparison_hours = [
        target_processing_hour - PREVIOUS_BATCH_OFFSET,
    ]

    comparison_hours += [
        target_processing_hour - offset
        for offset in HISTORY_OFFSETS
    ]

    comparison_hours = sorted(list(set(comparison_hours)))
    return comparison_hours


def build_dataset_for_target_batch_spark(
    source_df,
    target_processing_hour,
    min_rows_per_group=MIN_ROWS_PER_GROUP,
    max_rows_per_group=TARGET_SAMPLE_PER_GROUP,
    random_state=RANDOM_STATE
):
    positive = sample_by_processing_hour_spark(
        source_df=source_df,
        processing_hour=target_processing_hour,
        label=1,
        max_rows=max_rows_per_group,
        random_state=random_state
    )

    positive_rows = positive.count()
    if positive_rows < min_rows_per_group:
        return None, {
            "status": "skipped",
            "reason": "positive_rows_too_few",
            "target_processing_hour": str(target_processing_hour),
            "positive_rows": positive_rows,
        }

    comparison_hours = get_comparison_processing_hours(target_processing_hour)

    negative_parts = []
    negative_info = []

    for i, compare_hour in enumerate(comparison_hours):
        part = sample_by_processing_hour_spark(
            source_df=source_df,
            processing_hour=compare_hour,
            label=0,
            max_rows=max_rows_per_group,
            random_state=random_state + i + 100
        )

        rows = part.count()
        negative_info.append({"comparison_hour": str(compare_hour), "rows": rows})

        if rows >= min_rows_per_group:
            negative_parts.append(part)

    if len(negative_parts) == 0:
        return None, {
            "status": "skipped",
            "reason": "negative_rows_too_few",
            "target_processing_hour": str(target_processing_hour),
            "positive_rows": positive_rows,
            "negative_info": negative_info,
        }

    negative = negative_parts[0]
    for p in negative_parts[1:]:
        negative = negative.unionByName(p, allowMissingColumns=True)

    work = positive.unionByName(negative, allowMissingColumns=True)
    work = work.orderBy(F.rand(random_state)).cache()

    meta = {
        "status": "ok",
        "target_processing_hour": str(target_processing_hour),
        "positive_rows": int(work.filter(F.col("label") == 1).count()),
        "negative_rows": int(work.filter(F.col("label") == 0).count()),
        "total_rows": int(work.count()),
        "comparison_hours": [str(h) for h in comparison_hours],
        "negative_info": negative_info,
    }

    return work, meta


In [ ]:
def train_evaluate_one_batch_spark(
    batch_df,
    model_params,
    random_state=RANDOM_STATE,
    need_shap=False,
):
    result, artifacts = train_synapseml_on_work_df(
        batch_df,
        synapseml_params=model_params,
        random_state=random_state,
        shap_col=("featuresShap" if need_shap else None),
        return_artifacts=need_shap,
    )

    return result, artifacts


def compute_synapseml_shap_top_features(
    pred_df,
    feature_cols,
    target_processing_hour,
    top_n=10,
    sample_size=1000,
    random_state=RANDOM_STATE
):
    shap_top_df, shap_all_df = compute_top_shap_from_predictions(
        pred_df=pred_df,
        feature_cols=feature_cols,
        shap_col="featuresShap",
        top_n=top_n,
        sample_size=sample_size,
        seed=random_state
    )

    shap_top_pdf = shap_top_df.toPandas()
    shap_all_pdf = shap_all_df.toPandas()

    shap_top_pdf["target_processing_date_hour"] = str(target_processing_hour)
    shap_all_pdf["target_processing_date_hour"] = str(target_processing_hour)

    shap_top_pdf = shap_top_pdf.sort_values("mean_abs_shap", ascending=False)
    shap_top_pdf["rank"] = range(1, len(shap_top_pdf) + 1)

    shap_all_pdf = shap_all_pdf.sort_values("mean_abs_shap", ascending=False)
    shap_all_pdf["rank"] = range(1, len(shap_all_pdf) + 1)

    return shap_top_pdf, shap_all_pdf

In [ ]:
batch_auc_results = []
batch_shap_top_results = []
batch_shap_all_results = []
batch_run_logs = []

if RUN_BATCH_MONITORING:
    for idx, target_hour in enumerate(TARGET_BATCH_LIST):
        print("=" * 100)
        print(f"[{idx + 1}/{len(TARGET_BATCH_LIST)}] Running target batch:", target_hour)

        batch_df, meta = build_dataset_for_target_batch_spark(
            source_df=df_raw,
            target_processing_hour=target_hour,
            min_rows_per_group=MIN_ROWS_PER_GROUP,
            max_rows_per_group=TARGET_SAMPLE_PER_GROUP,
            random_state=RANDOM_STATE + idx
        )

        batch_run_logs.append(meta)

        if batch_df is None:
            print("Skipped:", meta["reason"])
            continue

        print("Rows:", meta["total_rows"])
        print("Positive rows:", meta["positive_rows"])
        print("Negative rows:", meta["negative_rows"])
        print("Comparison hours:", meta["comparison_hours"])

        try:
            eval_result, artifacts = train_evaluate_one_batch_spark(
                batch_df=batch_df,
                model_params=MODEL_PARAMS,
                random_state=RANDOM_STATE + idx,
                need_shap=RUN_BATCH_SHAP,
            )

            result_row = {
                "target_processing_date_hour": target_hour,
                "target_processing_date_hour_str": str(target_hour),
                "valid_auc": eval_result["valid_auc"],
                "test_auc": eval_result["test_auc"],
                "best_iteration": eval_result["best_iteration"],
                "total_rows": meta["total_rows"],
                "positive_rows": meta["positive_rows"],
                "negative_rows": meta["negative_rows"],
                "n_train": eval_result["n_train"],
                "n_valid": eval_result["n_valid"],
                "n_test": eval_result["n_test"],
                "comparison_hours": " | ".join(meta["comparison_hours"]),
            }

            batch_auc_results.append(result_row)

            print("Valid AUC:", eval_result["valid_auc"])
            print("Test AUC :", eval_result["test_auc"])
            print("Best iteration:", eval_result["best_iteration"])

            if RUN_BATCH_SHAP and artifacts is not None:
                shap_top_df, shap_all_df = compute_synapseml_shap_top_features(
                    pred_df=artifacts["test_pred_df"],
                    feature_cols=eval_result["feature_cols"],
                    target_processing_hour=str(target_hour),
                    top_n=10,
                    sample_size=1000,
                    random_state=RANDOM_STATE + idx
                )

                batch_shap_top_results.append(shap_top_df)
                batch_shap_all_results.append(shap_all_df)

                print("Top SHAP features:")
                display(shap_top_df[["rank", "feature", "mean_abs_shap"]])
            else:
                print("Batch SHAP skipped. Set RUN_BATCH_SHAP=True to compute SHAP for every batch.")

            batch_df.unpersist()
            gc.collect()

        except Exception as e:
            print("Failed target batch:", target_hour)
            print("Error:", repr(e))

            batch_run_logs.append({
                "status": "failed",
                "target_processing_hour": str(target_hour),
                "error": repr(e)
            })
else:
    print("Batch monitoring is disabled.")

In [ ]:
auc_series_df = pd.DataFrame(batch_auc_results)

if len(auc_series_df) > 0:
    auc_series_df = auc_series_df.sort_values("target_processing_date_hour").reset_index(drop=True)

    display(auc_series_df)

    auc_series_path = f"{OUTPUT_DIR}/batch_auc_series_synapseml.csv"
    auc_series_df.to_csv(auc_series_path, index=False)

    print("Saved:", auc_series_path)
else:
    print("No batch AUC results.")


In [ ]:
if len(auc_series_df) > 0:
    plt.figure(figsize=(14, 5))

    plt.plot(
        auc_series_df["target_processing_date_hour"],
        auc_series_df["test_auc"],
        marker="o",
        label="Test AUC"
    )

    plt.axhline(
        0.5,
        linestyle="--",
        label="AUC = 0.5"
    )

    plt.xlabel("Target processing_date_hour")
    plt.ylabel("Test AUC")
    plt.title("Batch-level Test AUC over Time - SynapseML")
    plt.xticks(rotation=45)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# Dynamic AUC monitoring with empirical p-value


In [ ]:
# Số điểm lịch sử tối thiểu trước khi bắt đầu đánh giá bất thường - GIỮ GIỐNG NOTEBOOK LIGHTGBM
MIN_HISTORY_POINTS = 20
ROLLING_WINDOW = None
ALPHA = 0.05

print("MIN_HISTORY_POINTS:", MIN_HISTORY_POINTS)
print("ROLLING_WINDOW:", ROLLING_WINDOW)
print("ALPHA:", ALPHA)


In [ ]:
def add_empirical_auc_anomaly_score(
    auc_df,
    auc_col="test_auc",
    time_col="target_processing_date_hour",
    min_history_points=6,
    rolling_window=None,
    alpha=0.05
):
    data = auc_df.copy()
    data = data.sort_values(time_col).reset_index(drop=True)

    p_values = []
    percentile_ranks = []
    history_means = []
    history_medians = []
    history_q95s = []
    history_sizes = []
    anomaly_flags = []
    anomaly_reasons = []

    for i, row in data.iterrows():
        current_auc = row[auc_col]
        history = data.loc[:i-1, auc_col].dropna()

        if rolling_window is not None:
            history = history.tail(rolling_window)

        history_size = len(history)
        history_sizes.append(history_size)

        if history_size < min_history_points or pd.isna(current_auc):
            p_values.append(np.nan)
            percentile_ranks.append(np.nan)
            history_means.append(np.nan)
            history_medians.append(np.nan)
            history_q95s.append(np.nan)
            anomaly_flags.append(False)
            anomaly_reasons.append("warmup_not_enough_history")
            continue

        p_value = (np.sum(history >= current_auc) + 1) / (history_size + 1)
        percentile_rank = (np.sum(history <= current_auc) + 1) / (history_size + 1)

        history_mean = history.mean()
        history_median = history.median()
        history_q95 = history.quantile(1 - alpha)

        is_anomaly = p_value <= alpha

        p_values.append(float(p_value))
        percentile_ranks.append(float(percentile_rank))
        history_means.append(float(history_mean))
        history_medians.append(float(history_median))
        history_q95s.append(float(history_q95))
        anomaly_flags.append(bool(is_anomaly))

        if is_anomaly:
            anomaly_reasons.append(f"auc_is_in_top_{int(alpha * 100)}pct_tail_of_history")
        else:
            anomaly_reasons.append("normal_vs_history")

    data["history_size"] = history_sizes
    data["history_auc_mean"] = history_means
    data["history_auc_median"] = history_medians
    data[f"history_auc_q{int((1-alpha)*100)}"] = history_q95s
    data["auc_empirical_p_value"] = p_values
    data["auc_percentile_rank"] = percentile_ranks
    data["is_auc_anomaly"] = anomaly_flags
    data["auc_anomaly_reason"] = anomaly_reasons

    return data


In [ ]:
if len(auc_series_df) > 0:
    auc_monitoring_df = add_empirical_auc_anomaly_score(
        auc_df=auc_series_df,
        auc_col="test_auc",
        time_col="target_processing_date_hour",
        min_history_points=MIN_HISTORY_POINTS,
        rolling_window=ROLLING_WINDOW,
        alpha=ALPHA
    )

    display(auc_monitoring_df)
else:
    auc_monitoring_df = pd.DataFrame()
    print("No AUC series for monitoring.")


In [ ]:
if len(auc_monitoring_df) > 0:
    auc_anomaly_batches_df = auc_monitoring_df[
        auc_monitoring_df["is_auc_anomaly"] == True
    ].copy()

    print("Number of AUC anomaly batches:", len(auc_anomaly_batches_df))

    display(
        auc_anomaly_batches_df[
            [
                "target_processing_date_hour",
                "test_auc",
                "history_size",
                "history_auc_mean",
                "history_auc_median",
                f"history_auc_q{int((1-ALPHA)*100)}",
                "auc_empirical_p_value",
                "auc_percentile_rank",
                "auc_anomaly_reason",
            ]
        ]
    )


In [ ]:
if len(auc_monitoring_df) > 0:
    q_col = f"history_auc_q{int((1-ALPHA)*100)}"

    plt.figure(figsize=(14, 5))

    plt.plot(
        auc_monitoring_df["target_processing_date_hour"],
        auc_monitoring_df["test_auc"],
        marker="o",
        label="Test AUC"
    )

    plt.plot(
        auc_monitoring_df["target_processing_date_hour"],
        auc_monitoring_df[q_col],
        marker="x",
        linestyle="--",
        label=f"Dynamic historical q{int((1-ALPHA)*100)}"
    )

    anomaly_points = auc_monitoring_df[auc_monitoring_df["is_auc_anomaly"] == True]

    plt.scatter(
        anomaly_points["target_processing_date_hour"],
        anomaly_points["test_auc"],
        s=120,
        marker="X",
        label="AUC anomaly"
    )

    plt.axhline(0.5, linestyle=":", label="AUC = 0.5")
    plt.xlabel("Target processing_date_hour")
    plt.ylabel("Test AUC")
    plt.title("Dynamic AUC Monitoring using Empirical Historical Distribution - SynapseML")
    plt.xticks(rotation=45)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
if len(auc_monitoring_df) > 0:
    plt.figure(figsize=(14, 5))

    plt.plot(
        auc_monitoring_df["target_processing_date_hour"],
        auc_monitoring_df["auc_empirical_p_value"],
        marker="o",
        label="Empirical p-value"
    )

    plt.axhline(ALPHA, linestyle="--", label=f"alpha = {ALPHA}")
    plt.xlabel("Target processing_date_hour")
    plt.ylabel("Empirical p-value")
    plt.title("AUC Empirical p-value over Time - SynapseML")
    plt.xticks(rotation=45)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
# Optional cleanup: chạy sau khi đã hoàn tất toàn bộ train/evaluate/monitoring.
for name in [
    "baseline_train_df",
    "valid_eval_df",
    "test_eval_df",
    "train_features_df",
    "valid_features_df",
    "test_features_df",
    "work_df",
]:
    obj = globals().get(name)
    try:
        if obj is not None:
            obj.unpersist()
            print("Unpersisted:", name)
    except Exception as e:
        print("Skip", name, "because:", repr(e))